In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql.types import *
spark = SparkSession.builder.appName("join").getOrCreate() 

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 19:17:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [15]:
#Reapontando um banco já criado, necessário no uso do docker sem o hive

spark.sql("""
CREATE DATABASE desp
LOCATION 'file:/opt/spark/apps/spark-warehouse/desp.db'
""")

DataFrame[]

In [21]:
#Reapontando uma tabela já criada, necessário no uso do docker sem o hive

spark.sql("""
CREATE TABLE despachantes
USING PARQUET
LOCATION 'file:/opt/spark/apps/spark-warehouse/desp.db/despachantes'
""")


DataFrame[]

In [23]:
spark.sql("select * from despachantes").show()

+---+-------------------+------+-------------+------+----------+
| id|               nome|status|       cidade|vendas|      data|
+---+-------------------+------+-------------+------+----------+
|  1|   Carminda Pestana| Ativo|  Santa Maria|    23|2020-08-11|
|  2|    Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
|  3|   Emídio Dornelles| Ativo| Porto Alegre|    34|2020-02-05|
|  4|Felisbela Dornelles| Ativo| Porto Alegre|    36|2020-02-05|
|  5|     Graça Ornellas| Ativo| Porto Alegre|    12|2020-02-05|
|  6|   Matilde Rebouças| Ativo| Porto Alegre|    22|2019-01-05|
|  7|    Noêmia   Orriça| Ativo|  Santa Maria|    45|2019-10-05|
|  8|      Roque Vásquez| Ativo| Porto Alegre|    65|2020-03-05|
|  9|      Uriel Queiroz| Ativo| Porto Alegre|    54|2018-05-05|
| 10|   Viviana Sequeira| Ativo| Porto Alegre|     0|2020-09-05|
+---+-------------------+------+-------------+------+----------+



In [24]:
#Criando data frame reclamacoes e 

recschema = "idrec INT, datarec STRING, iddesp INT"
reclamacoes = spark.read.csv("dados/reclamacoes.csv", header = False, schema = recschema)



In [26]:
#Persistindo no Banco 

reclamacoes.write.saveAsTable("reclamacoes")

In [27]:
#Criando dataframes para relacionamentos, via API

reclamacoes = spark.sql("select * from reclamacoes")
despachantes = spark.sql("select * from despachantes")


In [28]:
#Joins 

despachantes.join(reclamacoes, despachantes.id == reclamacoes.iddesp, "inner").select("idrec", "datarec" ,"iddesp","nome").show() 

+-----+----------+------+-------------------+
|idrec|   datarec|iddesp|               nome|
+-----+----------+------+-------------------+
|    1|2020-09-12|     2|    Deolinda Vilela|
|    2|2020-09-11|     2|    Deolinda Vilela|
|    3|2020-10-05|     4|Felisbela Dornelles|
|    4|2020-10-02|     5|     Graça Ornellas|
|    5|2020-12-06|     5|     Graça Ornellas|
|    6|2020-01-09|     5|     Graça Ornellas|
|    7|2020-01-05|     9|      Uriel Queiroz|
+-----+----------+------+-------------------+

